In [1]:
import requests
import json

url = "https://raw.githubusercontent.com/mempool/mining-pools/master/pools-v2.json"
response = requests.get(url)
pools = response.json()

print(f"Loaded {len(pools)} pools")
print(pools[0])  # peek at the shape of one entry

Loaded 171 pools
{'id': 1, 'name': 'BlockFills', 'addresses': ['1PzVut5X6Nx7Mv4JHHKPtVM9Jr9LJ4Rbry'], 'tags': ['/BlockfillsPool/'], 'link': 'https://www.blockfills.com/mining'}


In [2]:
address_to_pool = {}
for pool in pools:
    for addr in pool.get("addresses", []):
        address_to_pool[addr] = pool["name"]

print(f"Total known addresses across all pools: {len(address_to_pool)}")

Total known addresses across all pools: 218


In [4]:
import json
import pandas as pd

# Update this to your actual saved file path
with open("../data/coinbase_details2.json") as f:
    coinbase_data = json.load(f)

df = pd.DataFrame(coinbase_data)
df["block_timestamp"] = pd.to_datetime(df["block_timestamp"])
df["month"] = df["block_timestamp"].dt.to_period("M")

def match_pool(outputs_detail):
    for out in outputs_detail:
        addr = out["address"]
        if out["script_type"] != "nonstandard" and addr in address_to_pool:
            return address_to_pool[addr]
    return None  # no known pool matched this block's real address(es)

df["matched_pool"] = df["outputs_detail"].apply(match_pool)
df["attributed"] = df["matched_pool"].notna()

coverage = df.groupby("month")["attributed"].agg(total="count", attributed="sum")
coverage["coverage_pct"] = (coverage["attributed"] / coverage["total"] * 100).round(1)
print(coverage)

C:\Users\KwokYenMing\AppData\Local\Temp\ipykernel_13648\3824012090.py:10: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"] = df["block_timestamp"].dt.to_period("M")


         total  attributed  coverage_pct
month                                   
2012-06   4535          93           2.1
2018-06   4600        2384          51.8
2023-06   4313        2340          54.3


In [5]:
def get_real_address(outputs_detail):
    for out in outputs_detail:
        if out["script_type"] != "nonstandard":
            return out["address"]
    return None

unmatched = df[~df["attributed"]].copy()
unmatched["real_address"] = unmatched["outputs_detail"].apply(get_real_address)

for m in ["2018-06", "2023-06"]:
    print(f"\n--- {m} top unmatched addresses ---")
    print(unmatched[unmatched["month"].astype(str) == m]["real_address"].value_counts().head(10))


--- 2018-06 top unmatched addresses ---
real_address
1C1mCxRukix1KfegAY5zQQJV7samAciZpv    1229
18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX     422
1FVKW4rp5rN23dqFVk2tYGY4niAXMB8eZC      98
1RtUKxMRGBrz7Qt3YPZJb988PddKCNEFk       88
13TEThZNnKPk34HYAuo1QqYMwDdjF3qeHx      66
199RcwJ6iM1k7vkuYTWus5skMKDm1dP1r2      60
1Q7Jmho4FixWBiTVcZ5aKXv4rTMMp6CjiD      40
3JJfSJGBpTnaibi58ZPKxSL5xozQqLVSRA      40
14DjTuAUh87cwRsbU1z6W8hZY6FnEkpfLS      31
1EVzaFkkNNXq6RJh2oywwJMn8JPiq8ikDi      24
Name: count, dtype: int64

--- 2023-06 top unmatched addresses ---
real_address
38XnPvu9PmonFU9WouPXUjYbW91wa5MerL            937
18cBEMRxXHqzWWCxZNtU91F5sbUNKhL5PX            371
1Q8QR5k32hexiMQnRgkJ6fmmjn5fMWhdv9            283
1Mp82mJt6d8XzX8bN2GAkpDqBukiwLuKrA             70
3LwoXDToPiq2rBaQzoe2ZnaUvhGykFp2LM             65
33TbzA5AMiTKUCmeVEdsnTj3GiVXuavCAH             63
1DY2jW2bpsxpwfb5fHVHrgtAyoEVpU8hxZ             39
bc1qrpp7g75sx3ejclvsfdw2uahzchtyu7vumkuadu     28
bc1qte0s6pz7gsdlqq2cf6hv5mxcfksykyyy

In [7]:
len(df)

13448